# 00 — Overview: WESAD Preprocessing Pipeline

This folder turns the **synchronized raw** WESAD files (`WESAD/S*/S*.pkl`) into **cleaned, binary-labelled signals** using the methods from the `_forked` repository.

> The `S*.pkl` files are synchronized (chest RespiBAN ↔ wrist Empatica E4) but **not filtered**. `_forked` never saved a cleaned version — it filtered on-the-fly inside `compute_features()`. These notebooks extract that preprocessing into a separate stage.

## Stage → Notebook map

| Stage | Notebook | What it does | Original `_forked` source |
|---|---|---|---|
| 1. Load & inspect | `01_load_and_inspect.ipynb` | Load `S{id}.pkl`, show keys / shapes / sampling rates, plot raw signals & labels | `fb_code/feature_extraction.py` → `SubjectData.__init__`, `get_wrist_and_chest_data` |
| 2. Filter | `02_filter_signals.ipynb` | Per-sensor filtering, before/after plots | `fb_code/feature_extraction.py` → `compute_features`; `fb_code/utils.py` → `butter_lowpass_filter`, `filterSignalFIR` |
| 3. Label & save | `03_label_and_save.ipynb` | Binary relabel (stress vs non-stress), drop excluded samples, save all subjects | `fb_code/feature_extraction.py` → `label_dict`, `compute_features` groupby |

Helper code lives in **`wesad_preprocess.py`** (same folder), so the notebooks stay short.

## Per-sensor preprocessing (identical to `_forked`)

| Location | Sensor | fs (Hz) | Preprocessing | Source |
|---|---|---|---|---|
| Chest | ECG | 700 | passthrough (heartpy filters at feature stage) | `fb_code/ecg.py` |
| Chest | EDA | 700 | Butterworth LP 1 Hz, order 6 → standardize → `nk.eda_phasic` (SCR/SCL) | `compute_features` |
| Chest | EMG | 700 | passthrough | `compute_features` |
| Chest | Resp | 700 | passthrough (biosppy band-pass at feature stage) | `fb_code/respiration.py` |
| Chest | Temp | 700 | passthrough | `compute_features` |
| Chest | ACC | 700 | FIR LP 0.4 Hz, 64 taps | `utils.filterSignalFIR` |
| Wrist | BVP | 64 | passthrough | `compute_features` |
| Wrist | EDA | 4 | Butterworth LP 1 Hz, order 6 → standardize → `nk.eda_phasic` (SCR/SCL) | `compute_features` |
| Wrist | TEMP | 4 | passthrough | `compute_features` |
| Wrist | ACC | 32 | FIR LP 0.4 Hz, 64 taps | `utils.filterSignalFIR` |

## Faithfulness to `_forked`

The helper module is a **faithful port**: the same filters, the same parameters, and the same (causal) `lfilter` calls are used, so the output matches what `_forked` produced. Known quirks in the original code are intentionally kept — see the note at the bottom of this notebook.

## Labels

| WESAD id | Meaning | Binary |
|---|---|---|
| 1 | baseline | 0 (non-stress) |
| 2 | stress | 1 (stress) |
| 3 | amusement | 0 (non-stress) |
| 0, 4, 5, 6, 7 | transient / meditation / ignore | dropped |

## Output

`data/preprocessed/WESAD/S{id}_clean.pkl` (git-ignored), structure:

```python
{
  'subject': 'S2',
  'fs':     {'chest': {'ECG': 700, ...}, 'wrist': {'BVP': 64, ...}},
  'signal': {'chest': {'ECG': arr, 'EDA': arr, 'EDA_SCR': arr, 'EDA_SCL': arr, ...}, 'wrist': {...}},
  'label':  {'chest': {'ECG': arr(0/1), ...}, 'wrist': {...}},   # one label vector per signal, at its own rate
  'label_names': {0: 'non-stress', 1: 'stress'},
}
```

## Other `_forked` files (NOT used for WESAD)

| File | Purpose |
|---|---|
| `syncfcns.py`, `E4+Hexoskin_preprocess*.ipynb` | Raw E4 + Hexoskin sync for the RADWear dataset |
| `process_redcap.py`, `generate_metadata.py` | RADWear survey / metadata |
| `fb_code/noise.py`, `emd_noise/` | Adds synthetic noise (robustness experiments) |
| `fb_code/SQI.ipynb` | Signal quality index analysis |

## Environment check

In [1]:
import sys
from pathlib import Path

# make wesad_preprocess importable regardless of the notebook's working directory
HERE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'Scripts' / 'Preprocess').exists()) / 'Scripts' / 'Preprocess'
sys.path.insert(0, str(HERE))

import wesad_preprocess as wp

print('Repo root  :', wp.REPO_ROOT)
print('WESAD dir  :', wp.WESAD_DIR, '| exists:', wp.WESAD_DIR.exists())
print('Output dir :', wp.OUTPUT_DIR)
missing = [s for s in wp.SUBJECT_IDS if not (wp.WESAD_DIR / f'S{s}' / f'S{s}.pkl').exists()]
print('Missing .pkl files:', missing or 'none')

Repo root  : E:\Rizqy\Kuliah\Skripsi\repo-skripsi
WESAD dir  : E:\Rizqy\Kuliah\Skripsi\repo-skripsi\WESAD | exists: True
Output dir : E:\Rizqy\Kuliah\Skripsi\repo-skripsi\data\preprocessed\WESAD
Missing .pkl files: none


---

## Note — quirks in the original `_forked` code (kept as-is)

This pipeline is a faithful port of `_forked`, so the following observations are **documented, not fixed**. They only matter if you later question ACC results or compare against the existing `_forked` CSV files.

1. **ACC FIR loop filters across x/y/z, not along time** (`compute_features`):
   ```python
   for _ in acc_df.columns:
       acc_df[_] = utils.filterSignalFIR(acc_df.values)   # -> lfilter(FIR_coeff, 1, <(N,3) array>)
   ```
   `scipy.signal.lfilter` uses `axis=-1` by default, so on an (N, 3) array it filters each *row* of 3 values as if it were a short signal. Verified in this environment: all three axes come out as tiny scaled copies of raw `ACC_x` (`ACC_x = fir[0]·raw_x`, `fir[0] ≈ 0.00069`) with no temporal smoothing, and no warning is raised. This also applies to chest ACC (`acc_c_df`).

2. **FIR cutoff is normalized with 32 Hz for every ACC signal.** `filterSignalFIR` computes `f = cutoff / (fs_dict['ACC'] / 2)` with `fs_dict['ACC'] = 32` (wrist E4 rate), but it is also applied to chest ACC sampled at 700 Hz, making the effective cutoff ≈ 0.4 × 700/32 ≈ 8.75 Hz instead of 0.4 Hz.

3. **`lfilter` is causal** (phase delay) rather than zero-phase `filtfilt`. This is a design choice, not a bug — it is how `_forked` runs.

4. **Feature stage (not used here yet):** `get_peak_freq` hard-codes `fs=8` for every signal, and `get_dynamic_range = max/min` is unstable for signals that cross zero. Worth revisiting during feature extraction.

Because of 1–2, ACC values from this pipeline will match `_forked` but may not be what the FIR filter was originally intended to do. If you later decide to correct them, switch `filter_subject` to a per-axis, per-rate FIR call — but keep the default faithful unless the methodology says otherwise.